# 07 · Sensibilidad y fallas de producción (parte de JME)

Notebook de demostración de la parte 07 (`code/evaluate.py`, sección *Parte 07*). Guía:
`docs/instrucciones/07_sensibilidad_produccion.md`. El simulador supone que el resto del sistema es gratuito, instantáneo
y perfecto; aquí se mide cuánto cambia el resultado al relajar cada supuesto, **uno a la vez**.

Mientras la parte 05 no entregue el agente, los escenarios se corren con las líneas base *siempre largo* y *aleatorio*
para probar la maquinaria; `python main.py` los corre con el agente en cuanto existan sus políticas. La ablación y el
segundo algoritmo (pregunta 7) son de GC.

In [1]:
import sys
sys.path.insert(0, "..")
sys.modules.pop("code", None)  # IPython ya importó el módulo `code` estándar (ver docs/decisiones.md)

import pandas as pd

from code import data, env, evaluate
from main import BTC_CSV, FOLDS, SEED

ohlcv, _ = data.load_ohlcv(BTC_CSV)
features = data.compute_features(ohlcv)
pd.set_option("display.width", 200)

## 1. Los escenarios

Cada uno cambia un solo parámetro del entorno respecto a la base (stop primero, sin retraso, 0.125 % por lado):

In [2]:
fold = data.build_fold_dataset(ohlcv, features, 3, FOLDS[3])
latency = evaluate.latency_cost(fold.raw_features["train"])
pd.Series(evaluate.sensitivity_scenarios(env.COST_PER_SIDE, latency), name="parámetros del entorno").to_frame()

,parámetros del entorno
base,{}
tp_first,{'intrabar': 'tp_first'}
retraso_1_barra,{'obs_lag': 1}
latencia_2s,{'cost': 0.0013894538097869143}
costo_+5pb,{'cost': 0.00175}
costo_+10pb,{'cost': 0.0022500000000000003}
costo_+25pb,{'cost': 0.00375}


## 2. Latencia de 2 s como costo

El agente decide al cierre de t y se llena a la apertura de t + 1, que en Binance coincide con el cierre de t (mediana de
|apertura / cierre previo| ≈ 0 entre barras contiguas, parte 01). Si la inferencia tarda 2 s, se pierde el movimiento de
esos 2 s: con rendimientos sin deriva, su tamaño típico es σ̂·√(2/3600). Se agrega como costo extra por lado.

In [3]:
pd.DataFrame({k: {"σ̂ horaria mediana (train)": data.split_fold(features, FOLDS[k])["train"]["belief_vol"].median(),
                  "costo por lado en 2 s (pb)": 1e4 * evaluate.latency_cost(data.split_fold(features, FOLDS[k])["train"])}
              for k in FOLDS}).T.round(5)

,σ̂ horaria mediana (train),costo por lado en 2 s (pb)
1,0.00663,1.56256
2,0.00638,1.50293
3,0.00592,1.39454
4,0.00571,1.34641


## 3. Una prueba completa (fold 3, 2025) con siempre largo

In [4]:
table = evaluate.run_sensitivity(fold, lambda: evaluate.baseline_policies(3, SEED)["siempre_largo"],
                                FOLDS[3]["test"], SEED, env.COST_PER_SIDE)
table[["total_return", "sharpe", "max_drawdown", "n_trades", "cost_per_side", "n_ambiguous",
       "delta_total_return", "delta_sharpe"]].round(4)

,total_return,sharpe,max_drawdown,n_trades,cost_per_side,n_ambiguous,delta_total_return,delta_sharpe
scenario,,,,,,,,
base,-0.7596,-3.1692,-0.7731,505.0,0.0012,0.0,0.0000,0.0000
tp_first,-0.7596,-3.1692,-0.7731,505.0,0.0012,0.0,0.0000,0.0000
retraso_1_barra,-0.7437,-3.0511,-0.7619,535.0,0.0012,0.0,0.0159,0.1180
latencia_2s,-0.7912,-3.4968,-0.8025,505.0,0.0014,0.0,-0.0316,-0.3276
costo_+5pb,-0.8551,-4.3370,-0.8620,505.0,0.0018,0.0,-0.0955,-1.1678
costo_+10pb,-0.9127,-5.4844,-0.9161,505.0,0.0023,0.0,-0.1531,-2.3153
costo_+25pb,-0.9810,-8.7794,-0.9812,505.0,0.0038,0.0,-0.2214,-5.6103


## 4. Los cuatro folds (`artifacts/sensitivity.csv`)

Media y desviación entre folds del cambio contra la base. Con las líneas base, la resolución intrabarra casi no importa
(una sola operación ambigua en las cuatro pruebas): el resultado depende mucho más del costo de ejecución que del orden
de stop y take-profit dentro de la barra. El retraso del aleatorio mezcla el efecto del retraso con otro momento de
entrada al azar; con el agente (determinista) mide solo el retraso.

In [5]:
sens = pd.read_csv("../artifacts/sensitivity.csv", index_col=[0, 1, 2])
sens.loc[["media", "desv"], ["delta_total_return", "delta_sharpe"]].unstack(0).round(4)

delta_total_return         delta_sharpe        
fold                                       media    desv        media    desv
strategy      scenario                                                       
aleatorio     base                        0.0000  0.0000       0.0000  0.0000
              costo_+10pb                -0.1519  0.0554      -2.2856  0.4020
              costo_+25pb                -0.2535  0.1469      -5.6166  0.9436
              costo_+5pb                 -0.0898  0.0250      -1.1472  0.2045
              latencia_2s                -0.0296  0.0057      -0.3358  0.0791
              retraso_1_barra            -0.1123  0.1355      -1.1686  1.5022
              tp_first                    0.0011  0.0022       0.0138  0.0277
siempre_largo base                        0.0000  0.0000       0.0000  0.0000
              costo_+10pb                -0.2527  0.1481      -2.2325  0.4098
              costo_+25pb                -0.3827  0.1928      -5.5091  1.0564
              costo_+5pb                 -0.1556  0.0972      -1.1191  0.2021
              latencia_2s                -0.0546  0.0389      -0.3275  0.0766
              retraso_1_barra            -0.0136  0.0614      -0.0432  0.2818
              tp_first                    0.0000  0.0000       0.0000  0.0000